In [2]:
import rasterio
from rasterio.windows import Window
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from torch.utils.data.sampler import SubsetRandomSampler
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from segmentation_models_pytorch import Unet, UnetPlusPlus, DeepLabV3Plus
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import f1_score, matthews_corrcoef, accuracy_score, precision_score, recall_score, confusion_matrix, precision_recall_curve, auc, roc_curve
from sklearn.model_selection import StratifiedKFold, train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import os
import datetime
import sys
import pickle
from PIL import Image
import random
from pathlib import Path
import cv2
import gc

def get_next_model_iteration(base_path_model, base_iteration_name):
    """
    Automatically generate the next available model iteration name by incrementing the final number.
    
    Args:
        base_path_model: Base path where model directories are stored
        base_iteration_name: Base name like 'makassar_tallo_june2024_tires'
    
    Returns:
        str: Next available iteration name (e.g., 'makassar_tallo_june2024_tires_05')
    """
    import re
    
    # Extract the base name and current number
    # Pattern matches: base_name_XX where XX is a number
    pattern = r'^(.+?)_(\d+)$'
    match = re.match(pattern, base_iteration_name)
    
    if not match:
        # If no number at the end, start with _01
        base_name = base_iteration_name
        current_number = 0
    else:
        base_name = match.group(1)
        current_number = int(match.group(2))
    
    # Find the next available number
    next_number = current_number + 1
    
    # Check if the directory already exists, if so keep incrementing
    while True:
        next_iteration = f"{base_name}_{next_number:02d}"
        full_path = os.path.join(base_path_model, next_iteration)
        
        if not os.path.exists(full_path):
            break
        next_number += 1
    
    return next_iteration

class Config:
    def __init__(self):
        self.base_path = '/path/to/your/data'  # Root directory containing image and mask files
        self.base_path_model = self.base_path + '/unet_model_data/' # location for saving model data
        self.data_folder = self.base_path + '/drone_data_idn/' # location of images and masks
        self.image_folder = self.data_folder + 'drone_maps' # location of images - each image and mask pair should have the same prefix (e.g., 'place{A}_date{A}_image', 'place{A}_date{A}_mask', etc.)
        self.mask_folder = self.data_folder + 'masks_tires' # location of annotation masks
        self.image_suffix = '_image.tif'
        self.mask_suffix = '_mask_tires.tif'
        self.tile_size = 256
        self.overlap = 32
        self.batch_size = 32

        # Base iteration name (without the final number)
        self.base_iteration_name = 'makassar_tallo_june2024_tires'
        
        # DON'T call get_next_model_iteration here - we'll do it later when needed
        self.model_iteration = None  # Will be set later

        self.num_epochs = 3
        self.k_folds = 2 # number of folds
        self.random_seed = 42 # random seed for reproducibility
        self.initial_lr = 1e-4
        self.weight_decay = 1e-5
        self.train_threshold = 0.7
        self.val_threshold = 0.98
        self.model_def = 'UnetPlusPlus(encoder_name="efficientnet-b4", in_channels=3, classes=1, encoder_weights="imagenet")'
        
        # These will be set after model_iteration is determined
        self.initial_model_weights = None
        self.best_model_weights = None
        self.model_definition_file = None
        self.save_path = None
        self.checkpoint_path = None
        self.optimizer_path = None
        self.scheduler_path = None
        
        self.use_curriculum = True  # Whether to use curriculum learning
        self.use_negative_samples = True  # Set this to False to use only positive samples
        self.use_staged_curriculum = True  # Whether to use staged curriculum learning
        self.curriculum_start_ratio = 0.1  # Start with this proportion of negative samples to positive samples
        self.curriculum_end_ratio = 1.0  # End with this proportion of negative samples to positive
        self.curriculum_rate = 1.1  # Increase the ratio this much faster for validation set
        self.max_negative_ratio = self.curriculum_end_ratio
        self.curriculum_stages = 10  # Number of stages in the curriculum
        self.epochs_per_stage = 4   # Number of epochs to train on each stage
        self.holdout_percentage = 0.1  # 5% of the data for hold-out set

    def set_model_iteration(self, model_iteration):
        """Set the model iteration and update all related file paths."""
        self.model_iteration = model_iteration
        self.initial_model_weights = self.base_path_model + self.model_iteration + '/unet_trash_detection_initial.pth'
        self.best_model_weights = self.base_path_model + self.model_iteration + '/unet_trash_detection_best.pth'
        self.model_definition_file = self.base_path_model + self.model_iteration + '/model_definition.txt'
        self.save_path = self.base_path_model + self.model_iteration + '/unet_trash_detection.pth'
        self.checkpoint_path = self.base_path_model + self.model_iteration + '/checkpoint.pth'
        self.optimizer_path = self.base_path_model + self.model_iteration + '/optimizer.pth'
        self.scheduler_path = self.base_path_model + self.model_iteration + '/scheduler.pth'

# Create a config instance outside of any other class or function
config = Config()

def save_config_to_file(config, output_dir):
    """Save all config parameters to a plain text file for record-keeping."""
    config_file_path = os.path.join(output_dir, 'config_parameters.txt')
    
    with open(config_file_path, 'w') as f:
        f.write("Model Configuration Parameters\n")
        f.write("=" * 50 + "\n\n")
        f.write(f"Model Iteration: {config.model_iteration}\n")
        f.write(f"Timestamp: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        
        f.write("Data Paths:\n")
        f.write("-" * 20 + "\n")
        f.write(f"Base Path: {config.base_path}\n")
        f.write(f"Base Model Path: {config.base_path_model}\n")
        f.write(f"Data Folder: {config.data_folder}\n")
        f.write(f"Image Folder: {config.image_folder}\n")
        f.write(f"Mask Folder: {config.mask_folder}\n")
        f.write(f"Image Suffix: {config.image_suffix}\n")
        f.write(f"Mask Suffix: {config.mask_suffix}\n\n")
        
        f.write("Model Architecture:\n")
        f.write("-" * 20 + "\n")
        f.write(f"Model Definition: {config.model_def}\n\n")
        
        f.write("Training Parameters:\n")
        f.write("-" * 20 + "\n")
        f.write(f"Number of Epochs: {config.num_epochs}\n")
        f.write(f"K-Folds: {config.k_folds}\n")
        f.write(f"Random Seed: {config.random_seed}\n")
        f.write(f"Initial Learning Rate: {config.initial_lr}\n")
        f.write(f"Weight Decay: {config.weight_decay}\n")
        f.write(f"Batch Size: {config.batch_size}\n\n")
        
        f.write("Data Processing:\n")
        f.write("-" * 20 + "\n")
        f.write(f"Tile Size: {config.tile_size}\n")
        f.write(f"Overlap: {config.overlap}\n")
        f.write(f"Train Threshold: {config.train_threshold}\n")
        f.write(f"Validation Threshold: {config.val_threshold}\n\n")
        
        f.write("Curriculum Learning:\n")
        f.write("-" * 20 + "\n")
        f.write(f"Use Curriculum: {config.use_curriculum}\n")
        f.write(f"Use Negative Samples: {config.use_negative_samples}\n")
        f.write(f"Use Staged Curriculum: {config.use_staged_curriculum}\n")
        f.write(f"Curriculum Start Ratio: {config.curriculum_start_ratio}\n")
        f.write(f"Curriculum End Ratio: {config.curriculum_end_ratio}\n")
        f.write(f"Curriculum Rate: {config.curriculum_rate}\n")
        f.write(f"Max Negative Ratio: {config.max_negative_ratio}\n")
        f.write(f"Curriculum Stages: {config.curriculum_stages}\n")
        f.write(f"Epochs Per Stage: {config.epochs_per_stage}\n\n")
        
        f.write("Data Split:\n")
        f.write("-" * 20 + "\n")
        f.write(f"Holdout Percentage: {config.holdout_percentage}\n\n")
        
        f.write("File Paths:\n")
        f.write("-" * 20 + "\n")
        f.write(f"Initial Model Weights: {config.initial_model_weights}\n")
        f.write(f"Best Model Weights: {config.best_model_weights}\n")
        f.write(f"Model Definition File: {config.model_definition_file}\n")
        f.write(f"Save Path: {config.save_path}\n")
        f.write(f"Checkpoint Path: {config.checkpoint_path}\n")
        f.write(f"Optimizer Path: {config.optimizer_path}\n")
        f.write(f"Scheduler Path: {config.scheduler_path}\n")
    
    print(f"Configuration parameters saved to: {config_file_path}")

# Set random seed for reproducibility.  Use config.random_seed
random.seed(config.random_seed)
np.random.seed(config.random_seed)
torch.manual_seed(config.random_seed)
torch.cuda.manual_seed_all(config.random_seed)

def get_cached_stratification_labels(config, full_dataset):
    """
    Get stratification labels from cache if available, otherwise generate and cache them.
    
    Args:
        config: Configuration object
        full_dataset: The full dataset object
        
    Returns:
        tuple: (labels, cache_used) where cache_used is a boolean indicating if cache was used
    """
    import hashlib
    import pickle
    
    # Create cache directory if it doesn't exist
    cache_dir = os.path.join(config.base_path_model, 'stratification_cache')
    os.makedirs(cache_dir, exist_ok=True)
    
    # Create a hash of the tiling parameters and input data
    cache_key_data = {
        'tile_size': config.tile_size,
        'overlap': config.overlap,
        'image_files': [os.path.basename(pair[0]) for pair in full_dataset.image_mask_pairs],
        'mask_files': [os.path.basename(pair[1]) for pair in full_dataset.image_mask_pairs],
        'total_tiles': full_dataset.total_tiles,
        'n_tiles_width': full_dataset.n_tiles_width,
        'n_tiles_height': full_dataset.n_tiles_height
    }
    
    # Create a hash of the cache key data
    cache_key_str = str(sorted(cache_key_data.items()))
    cache_hash = hashlib.md5(cache_key_str.encode()).hexdigest()
    cache_file = os.path.join(cache_dir, f'stratification_labels_{cache_hash}.pkl')
    
    # Check if cache exists and is valid
    if os.path.exists(cache_file):
        try:
            with open(cache_file, 'rb') as f:
                cached_data = pickle.load(f)
                
            # Verify the cached data matches current parameters
            if (cached_data['cache_key'] == cache_key_data and 
                len(cached_data['labels']) == full_dataset.total_tiles):
                
                print(f"Using cached stratification labels from: {cache_file}")
                print(f"Cache info: {len(cached_data['labels'])} labels, "
                      f"positive samples: {np.sum(cached_data['labels'] == 1)}, "
                      f"negative samples: {np.sum(cached_data['labels'] == 0)}")
                return cached_data['labels'], True
            else:
                print("Cached stratification labels found but parameters don't match. Regenerating...")
        except Exception as e:
            print(f"Error reading cache file: {e}. Regenerating stratification labels...")
    
    # Generate new stratification labels
    print("Generating new stratification labels...")
    labels = []
    
    for i in tqdm(range(full_dataset.total_tiles), desc="Generating Stratification Labels"):
        try:
            _, mask = full_dataset[i]
            if torch.sum(mask == 1) > 0:
                labels.append(1)  # Trash present
            else:
                labels.append(0)  # No trash
        except Exception as e:
            print(f"Error processing item {i}: {str(e)}")
            labels.append(0)  # Treat as no trash
    
    labels = np.array(labels)
    
    # Cache the new labels
    try:
        cache_data = {
            'labels': labels,
            'cache_key': cache_key_data,
            'timestamp': datetime.datetime.now().isoformat(),
            'model_iteration': config.model_iteration
        }
        
        with open(cache_file, 'wb') as f:
            pickle.dump(cache_data, f)
        
        print(f"Stratification labels cached to: {cache_file}")
        print(f"Cache info: {len(labels)} labels, "
              f"positive samples: {np.sum(labels == 1)}, "
              f"negative samples: {np.sum(labels == 0)}")
        
    except Exception as e:
        print(f"Warning: Could not cache stratification labels: {e}")
    
    return labels, False

def check_existing_model_iteration(config, full_dataset):
    """
    Check if there's an existing model iteration with matching tiling parameters.
    We always create new directories, but can reuse cache files.
    
    Args:
        config: Configuration object
        full_dataset: The full dataset object
        
    Returns:
        tuple: (cache_available, cache_file_path) where cache_available indicates if cache can be reused
    """
    # Look for existing model directories with matching parameters
    for item in os.listdir(config.base_path_model):
        if item.startswith(config.base_iteration_name) and os.path.isdir(os.path.join(config.base_path_model, item)):
            # Check if this directory has a config file
            config_file = os.path.join(config.base_path_model, item, 'config_parameters.txt')
            if os.path.exists(config_file):
                try:
                    # Read the config file to check parameters
                    with open(config_file, 'r') as f:
                        config_content = f.read()
                    
                    # Check if tile size and overlap match
                    if (f"Tile Size: {config.tile_size}" in config_content and 
                        f"Overlap: {config.overlap}" in config_content):
                        
                        # Check if cache file exists with matching parameters
                        cache_dir = os.path.join(config.base_path_model, 'stratification_cache')
                        if os.path.exists(cache_dir):
                            # Create the same cache key that would be used for current dataset
                            cache_key_data = {
                                'tile_size': config.tile_size,
                                'overlap': config.overlap,
                                'image_files': [os.path.basename(pair[0]) for pair in full_dataset.image_mask_pairs],
                                'mask_files': [os.path.basename(pair[1]) for pair in full_dataset.image_mask_pairs],
                                'total_tiles': full_dataset.total_tiles,
                                'n_tiles_width': full_dataset.n_tiles_width,
                                'n_tiles_height': full_dataset.n_tiles_height
                            }
                            
                            # Look for cache files that match this directory's parameters
                            for cache_file in os.listdir(cache_dir):
                                if cache_file.startswith('stratification_labels_') and cache_file.endswith('.pkl'):
                                    cache_path = os.path.join(cache_dir, cache_file)
                                    try:
                                        with open(cache_path, 'rb') as f:
                                            cached_data = pickle.load(f)
                                        # Check if this cache matches current parameters
                                        if (cached_data['cache_key'] == cache_key_data):
                                            print(f"Found cache file with matching tiling parameters from: {item}")
                                            print("Will reuse cache file for faster evaluation...")
                                            return True, cache_path
                                    except Exception:
                                        continue
                        
                        print(f"Found directory {item} with matching tiling parameters but no cache file.")
                        break
                        
                except Exception as e:
                    print(f"Error reading config file for {item}: {e}")
                    continue
    
    print("No existing cache found. Will generate new stratification labels.")
    return False, None

# --- Data Handling ---
def get_image_mask_pairs(config):
    image_files = [f for f in os.listdir(config.image_folder) if f.endswith(config.image_suffix)]
    pairs = []
    
    for image_file in image_files:
        base_name = image_file[:-len(config.image_suffix)]
        mask_file = base_name + config.mask_suffix
        mask_path = os.path.join(config.mask_folder, mask_file)
        
        if os.path.exists(mask_path):
            pairs.append((
                os.path.join(config.image_folder, image_file),
                mask_path
            ))
        else:
            print(f"Warning: No matching mask found for {image_file}")
    
    # print(f"Found {len(pairs)} image-mask pairs.")
    # for i, (img, mask) in enumerate(pairs):
    #     print(f"Pair {i+1}: Image: {os.path.basename(img)}, Mask: {os.path.basename(mask)}")
    
    return pairs

class DroneTrashDataset(Dataset):
    def __init__(self, config, transforms, mode='train', indices=None):
        self.config = config
        self.transforms = transforms
        self.mode = mode
        
        self.image_mask_pairs = get_image_mask_pairs(config)
        self.image_srcs = [rasterio.open(img_path) for img_path, _ in self.image_mask_pairs]
        self.mask_srcs = [rasterio.open(mask_path) for _, mask_path in self.image_mask_pairs]
        
        self.total_tiles = 0
        self.tile_offsets = [0]
        self.n_tiles_width = []
        self.n_tiles_height = []
        
        for img_src in self.image_srcs:
            image_width = img_src.width
            image_height = img_src.height
            n_tiles_w = (image_width + config.tile_size - config.overlap - 1) // (config.tile_size - config.overlap)
            n_tiles_h = (image_height + config.tile_size - config.overlap - 1) // (config.tile_size - config.overlap)
            n_tiles = n_tiles_w * n_tiles_h
            self.total_tiles += n_tiles
            self.tile_offsets.append(self.total_tiles)
            self.n_tiles_width.append(n_tiles_w)
            self.n_tiles_height.append(n_tiles_h)

        if indices is None:
            self.indices = list(range(self.total_tiles))
        else:
            self.indices = indices

        print(f"{self.mode.capitalize()} dataset - Total tiles: {self.total_tiles}, Valid tiles: {len(self.indices)}")

    def filter_tiles(self):
        valid_indices = []
        with tqdm(total=self.total_tiles, desc="Filtering tiles") as pbar:
            for idx in range(self.total_tiles):
                mask_tile = self.get_mask_tile(idx)
                if not np.all(mask_tile == 255):  # Exclude if all pixels are NoData
                    valid_indices.append(idx)
                pbar.update(1)
                if idx % 1000 == 0 or idx == self.total_tiles - 1:
                    pbar.set_postfix(valid_tiles=len(valid_indices))
        return valid_indices

    def get_mask_tile(self, idx):
        img_idx = next(i for i, offset in enumerate(self.tile_offsets) if offset > idx) - 1
        local_idx = idx - self.tile_offsets[img_idx]
        mask_src = self.mask_srcs[img_idx]
        n_tiles_width = self.n_tiles_width[img_idx]
        row = local_idx // n_tiles_width
        col = local_idx % n_tiles_width
        x_offset = col * (self.config.tile_size - self.config.overlap)
        y_offset = row * (self.config.tile_size - self.config.overlap)
        mask_tile = mask_src.read(
            1,
            window=Window(x_offset, y_offset, self.config.tile_size, self.config.tile_size)
        )
        return mask_tile

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        if not isinstance(idx, (int, np.integer)):
            raise TypeError(f"Index must be an integer, not {type(idx)}")

        try:
            real_idx = self.indices[idx]
            
            # Find which image/mask pair this index belongs to
            img_idx = next(i for i, offset in enumerate(self.tile_offsets) if offset > real_idx) - 1
            
            local_idx = real_idx - self.tile_offsets[img_idx]

            # print(f"Getting item {idx}, real_idx: {real_idx}, img_idx: {img_idx}, local_idx: {local_idx}")
            
            image_src = self.image_srcs[img_idx]
            mask_src = self.mask_srcs[img_idx]
            
            n_tiles_width = self.n_tiles_width[img_idx]
            
            row = local_idx // n_tiles_width
            col = local_idx % n_tiles_width
            
            x_offset = col * (self.config.tile_size - self.config.overlap)
            y_offset = row * (self.config.tile_size - self.config.overlap)
           
            # Read image tile
            image_tile = image_src.read(
                [1, 2, 3],
                window=Window(x_offset, y_offset, self.config.tile_size, self.config.tile_size)
            )
            image_tile = np.transpose(image_tile, (1, 2, 0))  # CHW to HWC
            image_tile = image_tile.astype(np.float32) / 255.0  # Scale to [0, 1]
            
            # Read mask tile
            mask_tile = mask_src.read(
                1,
                window=Window(x_offset, y_offset, self.config.tile_size, self.config.tile_size)
            )
            mask_tile = mask_tile.squeeze()  # Remove the channel dimension
            
            # Handle NoData and binarize mask
            mask_tile = np.where(mask_tile == 255, 0, mask_tile)
            mask_tile = np.where(mask_tile > 0, 1, 0).astype(np.float32)

            # Apply transformations
            if self.transforms:
                if self.mode == 'train':
                    # Concatenate image and mask
                    combined = np.concatenate([image_tile, mask_tile[..., None]], axis=-1)
                    # Apply spatial transformations
                    augmented = self.transforms['spatial'](image=combined)
                    combined_transformed = augmented['image']

                    # Split the combined array back into image and mask
                    image_tile = combined_transformed[..., :3]
                    mask_tile = combined_transformed[..., 3]
                    mask_tile = np.where(mask_tile > 0, 1, 0)  # ensure the masks remain binary

                    # Convert to uint8 for photometric transformations
                    image_tile = (image_tile * 255).astype(np.uint8)
        
                    # Apply photometric transformations to the image only
                    image_tile = self.transforms['photometric'](image=image_tile)['image']
        
                    # Convert back to float32
                    image_tile = image_tile.astype(np.float32) / 255.0
        
                    # Apply normalization to the image
                    # image_tile = self.transforms['normalization'](image=image_tile)['image']

                elif self.mode == 'subset':
                    # Apply only resize for subsetting
                    augmented = self.transforms(image=image_tile, mask=mask_tile)
                    image_tile = augmented['image']
                    mask_tile = augmented['mask']
                else:  # val, test, base, or any other mode
                    # Apply resize and normalization to image, only resize to mask
                    image_tile = self.transforms['val_image'](image=image_tile)['image']
                    mask_tile = self.transforms['val_mask'](image=mask_tile)['image']

            # Ensure mask is binary
            mask_tile = np.where(mask_tile > 0, 1, 0).astype(np.float32)

            # Convert to tensor if not already
            if not isinstance(image_tile, torch.Tensor):
                image_tile = torch.from_numpy(image_tile).float()
            if not isinstance(mask_tile, torch.Tensor):
                mask_tile = torch.from_numpy(mask_tile).float()

            # Ensure correct shapes
            image_tile = image_tile.permute(2, 0, 1) if image_tile.ndim == 3 else image_tile
            mask_tile = mask_tile.unsqueeze(0) if mask_tile.ndim == 2 else mask_tile

            return image_tile, mask_tile

        except Exception as e:
            print(f"Error processing item {idx}: {str(e)}")
            return None

    def __del__(self):
        for src in self.image_srcs + self.mask_srcs:
            src.close()

def get_transforms(config):
    spatial_transforms = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=30, p=0.5),
        # A.RandomScale(scale_limit=0.1, p=0.5),
        # A.ElasticTransform(alpha=0.1, sigma=10, alpha_affine=10, p=0.5),
        A.Resize(config.tile_size, config.tile_size, always_apply=True),
    ])

    photometric_transforms = A.Compose([
        # A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        # A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=10, val_shift_limit=5, p=0.5),
        # A.GaussNoise(var_limit=(5.0, 25.0), p=0.2)
    ])

    # normalization_transform = A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)) # either normalize OR scale (x/255), not both!

    val_image_transform = A.Compose([
        A.Resize(config.tile_size, config.tile_size, always_apply=True),
        # normalization_transform,
    ])

    val_mask_transform = A.Resize(config.tile_size, config.tile_size, always_apply=True)

    return {
        'spatial': spatial_transforms,
        'photometric': photometric_transforms,
        # 'normalization': normalization_transform,
        'val_image': val_image_transform,
        'val_mask': val_mask_transform
    }


class CurriculumDataset(Dataset):
    def __init__(self, dataset, positive_indices, negative_indices, num_epochs, mode='train', config=None):
        self.dataset = dataset
        self.positive_indices = positive_indices
        self.negative_indices = negative_indices
        self.num_epochs = num_epochs
        self.current_epoch = 0
        self.mode = mode
        self.config = config
        self.update_epoch(0)

    def update_epoch(self, epoch):
        self.current_epoch = epoch
        num_positives = len(self.positive_indices)
        num_negatives = 0  # Initialize num_negatives to 0
    
        if self.config.use_negative_samples:
            if self.config.use_curriculum:
                if self.config.use_staged_curriculum:
                    stage = min(epoch // self.config.epochs_per_stage, self.config.curriculum_stages - 1)
                    progress = stage / (self.config.curriculum_stages - 1)
                else:
                    progress = epoch / (self.num_epochs - 1) if self.num_epochs > 1 else 1
    
                if self.mode == 'train':
                    ratio = self.config.curriculum_start_ratio + (self.config.curriculum_end_ratio - self.config.curriculum_start_ratio) * progress
                else:  # validation
                    ratio = self.config.curriculum_start_ratio + (self.config.curriculum_end_ratio - self.config.curriculum_start_ratio) * progress * self.config.curriculum_rate
            else:
                ratio = self.config.max_negative_ratio
    
            num_negatives = int(ratio * num_positives)
            num_negatives = min(num_negatives, len(self.negative_indices))
            self.indices = self.positive_indices + random.sample(self.negative_indices, num_negatives)
        else:
            self.indices = self.positive_indices
    
        random.shuffle(self.indices)
        
        if num_positives > 0:
            ratio = num_negatives / num_positives
        else:
            ratio = 0
        
        print(f"Epoch {epoch+1} ({self.mode.capitalize()}): {num_positives} positive, {num_negatives} negative samples (ratio: {ratio:.2f})")

    def __getitem__(self, idx):
        return self.dataset[self.indices[idx]]

    def __len__(self):
        return len(self.indices)


class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=1.0, dice_weight=1.0, pos_weight=None): # adjust bce and dice weights if one should be more influential
        super(BCEDiceLoss, self).__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.bce_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight) if pos_weight is not None else nn.BCEWithLogitsLoss()

    def forward(self, inputs, targets):
        # BCE Loss
        bce = self.bce_loss(inputs, targets)

        # Dice Loss
        inputs = torch.sigmoid(inputs)  # Apply sigmoid since BCEWithLogits expects raw logits
        smooth = 1.0  # Smoothing constant to prevent division by zero
        inputs_flat = inputs.view(-1)
        targets_flat = targets.view(-1)
        
        intersection = (inputs_flat * targets_flat).sum()
        dice = 1 - ((2. * intersection + smooth) / (inputs_flat.sum() + targets_flat.sum() + smooth))

        # Combined loss
        combined_loss = (self.bce_weight * bce) + (self.dice_weight * dice)
        return combined_loss

# set the device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Weighted Loss Function
pos_weight = torch.tensor([20.0]).to(device)  # Adjust pos_weight for imbalance (trash tires training (filtered to postive masks) set has about 5% of pixels labeled as trash (1/20th))
criterion = BCEDiceLoss(bce_weight=1.0, dice_weight=1.0, pos_weight=pos_weight).to(device) # For a single-class predictor; adjust weights if needed

# create model using config.model_def and eval()
model = eval(config.model_def).to(device)

def train_model(model, config, train_loader, val_loader, criterion, optimizer, scheduler):
    epoch_train_losses = []
    epoch_train_f1_scores = []
    epoch_train_mcc_scores = []
    epoch_train_accuracies = []
    epoch_train_precisions = []
    epoch_train_recalls = []
    epoch_val_losses = []
    epoch_val_f1_scores = []
    epoch_val_mcc_scores = []
    epoch_val_accuracies = []
    epoch_val_precisions = []
    epoch_val_recalls = []
    
    best_mcc = -1.0
    best_model_state = None
    best_loss = float('inf')
    best_epoch = -1

    try:
        for epoch in range(config.num_epochs):
            if isinstance(train_loader.dataset, CurriculumDataset):
                train_loader.dataset.update_epoch(epoch)
            if isinstance(val_loader.dataset, CurriculumDataset):
                val_loader.dataset.update_epoch(epoch)
            
            model.train()
            running_loss = 0.0
            running_f1 = 0.0
            running_mcc = 0.0
            running_accuracy = 0.0
            running_precision = 0.0
            running_recall = 0.0
            total_batches = len(train_loader)

            for images, masks in train_loader:
                if images is None or masks is None:
                    continue
    
                images = images.float().to(device)
                masks = masks.float().to(device)
    
                if masks.ndim == 3:
                    masks = masks.unsqueeze(1)
    
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, masks)
                loss.backward()
                optimizer.step()
    
                running_loss += loss.item() * images.size(0)
                preds = (outputs > config.train_threshold).float()
                accuracy, precision, recall, f1, mcc = calculate_metrics(masks.cpu().numpy().flatten(), preds.cpu().numpy().flatten())
                running_f1 += f1
                running_mcc += mcc
                running_accuracy += accuracy
                running_precision += precision
                running_recall += recall
    
            epoch_loss = running_loss / len(train_loader.dataset)
            epoch_f1 = running_f1 / total_batches
            epoch_mcc = running_mcc / total_batches
            epoch_accuracy = running_accuracy / total_batches
            epoch_precision = running_precision / total_batches
            epoch_recall = running_recall / total_batches
    
            epoch_train_losses.append(epoch_loss)
            epoch_train_f1_scores.append(epoch_f1)
            epoch_train_mcc_scores.append(epoch_mcc)
            epoch_train_accuracies.append(epoch_accuracy)
            epoch_train_precisions.append(epoch_precision)
            epoch_train_recalls.append(epoch_recall)
    
            # First, evaluate the model to get validation metrics
            val_loss, val_accuracy, val_precision, val_recall, val_f1, val_mcc = evaluate_model(
                model, val_loader, criterion, config, save_plots=False,  # Don't save plots yet
                plot_dir=None, evaluation_type="validation"
            )
    
            epoch_val_losses.append(val_loss)
            epoch_val_f1_scores.append(val_f1)
            epoch_val_mcc_scores.append(val_mcc)
            epoch_val_accuracies.append(val_accuracy)
            epoch_val_precisions.append(val_precision)
            epoch_val_recalls.append(val_recall)
    
            scheduler.step(val_loss)
            current_lr = scheduler.get_last_lr()[0]
    
            # Check if this is the best epoch
            is_best_epoch = val_mcc > best_mcc or (val_mcc == best_mcc and val_loss < best_loss)
            
            if is_best_epoch:
                best_mcc = val_mcc
                best_loss = val_loss
                best_model_state = model.state_dict().copy()
                best_epoch = epoch + 1  # +1 because epoch is 0-indexed
                tqdm.write(f"New best model at epoch {best_epoch} with MCC: {best_mcc:.6f} and loss: {best_loss:.6f}")
                
                # Now save plots for the best epoch
                evaluate_model(
                    model, val_loader, criterion, config, save_plots=True,
                    plot_dir=f'{config.base_path_model}{config.model_iteration}',
                    evaluation_type="validation", is_best_epoch=True
                )
    
            tqdm.write(f"Epoch {epoch+1}/{config.num_epochs} - "
                       f"Train Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}, "
                       f"Train F1: {epoch_f1:.4f}, Val F1: {val_f1:.4f}, "
                       f"Train MCC: {epoch_mcc:.4f}, Val MCC: {val_mcc:.4f}, "
                       f"Train Precision: {epoch_precision:.4f}, Val Precision: {val_precision:.4f}, "
                       f"Train Recall: {epoch_recall:.4f}, Val Recall: {val_recall:.4f}, "
                       f"LR: {current_lr:.6f}")
        
        return (epoch_train_losses, epoch_train_f1_scores, epoch_train_mcc_scores, epoch_train_accuracies,
                epoch_train_precisions, epoch_train_recalls,
                epoch_val_losses, epoch_val_f1_scores, epoch_val_mcc_scores, epoch_val_accuracies,
                epoch_val_precisions, epoch_val_recalls,
                best_mcc, best_model_state, best_epoch)
        
    except KeyboardInterrupt:
        tqdm.write("\nTraining interrupted by user. Saving current state...")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
        }, f'{config.base_path_model}{config.model_iteration}/interrupted_checkpoint.pth')
    except Exception as e:
        tqdm.write(f"\nUnexpected error during training: {e}")
        raise e

def calculate_f1(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return f1

def calculate_metrics(y_true, y_pred):
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0

    return accuracy, precision, recall, f1, mcc


def evaluate_model(model, val_loader, criterion, config, pbar=None, save_plots=False, plot_dir=None, evaluation_type="validation", is_best_epoch=False):
    """
    Memory-efficient evaluation function that computes metrics using confusion matrix accumulation.
    
    Args:
        model: PyTorch model to evaluate
        val_loader: DataLoader for validation data
        criterion: Loss function
        config: Configuration object
        pbar: Progress bar (optional)
        save_plots: Whether to save performance plots
        plot_dir: Directory to save plots
        evaluation_type: Type of evaluation (train/val/holdout)
        is_best_epoch: Whether this is the best epoch (for saving plots)
        
    Returns:
        tuple: (loss, accuracy, precision, recall, f1, mcc)
    """
    model.eval()
    total_loss = 0.0
    num_batches = 0
    
    # Initialize confusion matrix (2x2 for binary classification)
    confusion_matrix = np.zeros((2, 2), dtype=np.int64)
    
    # For ROC/PR curves, we only need to store minimal data
    all_scores = []
    all_labels = []
    
    with torch.no_grad():
        for batch_idx, (images, masks) in enumerate(val_loader):
            try:
                images = images.to(device)
                masks = masks.to(device)
                
                outputs = model(images)
                loss = criterion(outputs, masks)
                total_loss += loss.item()
                num_batches += 1
                
                # Convert to numpy for metric calculation
                preds = (torch.sigmoid(outputs) > 0.5).float()
                batch_masks = masks.cpu().numpy().flatten()
                batch_preds = preds.cpu().numpy().flatten()
                batch_scores = torch.sigmoid(outputs).cpu().numpy().flatten()
                
                # Update confusion matrix directly (most memory-efficient)
                for i in range(len(batch_masks)):
                    true_label = int(batch_masks[i])
                    pred_label = int(batch_preds[i])
                    confusion_matrix[true_label, pred_label] += 1
                
                # Store only scores and labels for ROC/PR curves (minimal memory)
                if save_plots and is_best_epoch:
                    all_scores.extend(batch_scores)
                    all_labels.extend(batch_masks)
                
                # Clear GPU cache after each batch
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                
                if pbar:
                    pbar.set_postfix({'val_loss': f'{loss.item():.4f}'})
                    
            except Exception as e:
                tqdm.write(f"\nError processing batch {batch_idx}: {e}")
                continue
    
    # Calculate metrics from confusion matrix
    tn, fp, fn, tp = [float(x) for x in confusion_matrix.ravel()]

    # Avoid division by zero
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    # MCC calculation — use split sqrt to avoid int64 overflow on large pixel counts
    mcc_num = tp * tn - fp * fn
    mcc_den = np.sqrt(tp + fp) * np.sqrt(tp + fn) * np.sqrt(tn + fp) * np.sqrt(tn + fn)
    mcc = mcc_num / mcc_den if mcc_den > 0 else 0
    
    avg_loss = total_loss / num_batches if num_batches > 0 else 0
    
    # Save plots only if requested and this is the best epoch
    if save_plots and is_best_epoch and plot_dir:
        try:
            # Convert to numpy arrays only when needed for plotting
            if all_scores and all_labels:
                scores_array = np.array(all_scores)
                labels_array = np.array(all_labels)
                
                _save_performance_plots(
                    labels_array, 
                    None,  # We don't need predictions for confusion matrix plots
                    scores_array, 
                    confusion_matrix, 
                    plot_dir, 
                    config.model_iteration, 
                    evaluation_type
                )
                
                # Clear arrays immediately after use
                del scores_array, labels_array
                
        except Exception as e:
            tqdm.write(f"\nWarning: Could not save performance plots: {e}")
    
    # Clear accumulated data
    del all_scores, all_labels
    
    # Force garbage collection
    gc.collect()
    
    return avg_loss, accuracy, precision, recall, f1, mcc


def _save_performance_plots(targets, predictions, probabilities, confusion_mat, plot_dir, model_iteration, evaluation_type):
    """
    Memory-efficient helper function to create and save performance visualization plots.
    
    Args:
        targets: Ground truth labels (only needed for ROC/PR curves)
        predictions: Predicted labels (can be None if using confusion matrix)
        probabilities: Predicted probabilities for ROC/PR curves
        confusion_mat: Confusion matrix for visualization
        plot_dir: Directory to save plots
        model_iteration: Model iteration name
        evaluation_type: Type of evaluation
    """
    try:
        # Create plots directory
        plots_dir = os.path.join(plot_dir, 'performance_plots')
        os.makedirs(plots_dir, exist_ok=True)
        
        # 1. Confusion Matrix Plot
        plt.figure(figsize=(8, 6), dpi=150)
        sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues', 
                   xticklabels=['Negative', 'Positive'], 
                   yticklabels=['Negative', 'Positive'])
        plt.title(f'Confusion Matrix - {evaluation_type.capitalize()} Set\n{model_iteration}')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        plt.savefig(os.path.join(plots_dir, f'confusion_matrix_{evaluation_type}.png'), dpi=150, bbox_inches='tight')
        plt.close()
        
        # 2. ROC Curve (only if we have probabilities)
        if probabilities is not None and targets is not None:
            plt.figure(figsize=(8, 6), dpi=150)
            fpr, tpr, _ = roc_curve(targets, probabilities)
            auc_score = auc(fpr, tpr)
            
            plt.plot(fpr, tpr, color='darkorange', lw=2, 
                    label=f'ROC curve (AUC = {auc_score:.3f})')
            plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
            plt.xlim([0.0, 1.0])
            plt.ylim([0.0, 1.05])
            plt.xlabel('False Positive Rate')
            plt.ylabel('True Positive Rate')
            plt.title(f'ROC Curve - {evaluation_type.capitalize()} Set\n{model_iteration}')
            plt.legend(loc="lower right")
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(os.path.join(plots_dir, f'roc_curve_{evaluation_type}.png'), dpi=150, bbox_inches='tight')
            plt.close()
            
            # 3. Precision-Recall Curve
            plt.figure(figsize=(8, 6), dpi=150)
            precision, recall, _ = precision_recall_curve(targets, probabilities)
            pr_auc = auc(recall, precision)
            
            plt.plot(recall, precision, color='blue', lw=2, 
                    label=f'PR curve (AUC = {pr_auc:.3f})')
            plt.xlabel('Recall')
            plt.ylabel('Precision')
            plt.title(f'Precision-Recall Curve - {evaluation_type.capitalize()} Set\n{model_iteration}')
            plt.legend(loc="lower left")
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(os.path.join(plots_dir, f'precision_recall_curve_{evaluation_type}.png'), dpi=150, bbox_inches='tight')
            plt.close()
        
        # Force cleanup
        plt.close('all')
        gc.collect()
        
    except Exception as e:
        print(f"Error saving performance plots: {e}")
        plt.close('all')

# Plotting function to visualize validation loss and F1 scores across the epochs
def plot_cv_results(train_losses, train_f1_scores, train_mcc_scores, train_accuracies,
                    train_precisions, train_recalls,
                    val_losses, val_f1_scores, val_mcc_scores, val_accuracies,
                    val_precisions, val_recalls,
                    num_folds, num_epochs, config):
    
    # Reshape the data
    train_losses = np.array(train_losses).reshape(num_folds, num_epochs)
    val_losses = np.array(val_losses).reshape(num_folds, num_epochs)
    train_f1_scores = np.array(train_f1_scores).reshape(num_folds, num_epochs)
    val_f1_scores = np.array(val_f1_scores).reshape(num_folds, num_epochs)
    train_mcc_scores = np.array(train_mcc_scores).reshape(num_folds, num_epochs)
    val_mcc_scores = np.array(val_mcc_scores).reshape(num_folds, num_epochs)
    train_accuracies = np.array(train_accuracies).reshape(num_folds, num_epochs)
    val_accuracies = np.array(val_accuracies).reshape(num_folds, num_epochs)
    train_precisions = np.array(train_precisions).reshape(num_folds, num_epochs)
    val_precisions = np.array(val_precisions).reshape(num_folds, num_epochs)
    train_recalls = np.array(train_recalls).reshape(num_folds, num_epochs)
    val_recalls = np.array(val_recalls).reshape(num_folds, num_epochs)

    epochs = range(1, num_epochs + 1)

    plt.figure(figsize=(15, 30))

    # Loss plot
    plt.subplot(6, 1, 1)
    plot_metric(epochs, train_losses, val_losses, "Loss")

    # F1 Score plot
    plt.subplot(6, 1, 2)
    plot_metric(epochs, train_f1_scores, val_f1_scores, "F1 Score")

    # MCC plot
    plt.subplot(6, 1, 3)
    plot_metric(epochs, train_mcc_scores, val_mcc_scores, "MCC")

    # Accuracy plot
    plt.subplot(6, 1, 4)
    plot_metric(epochs, train_accuracies, val_accuracies, "Accuracy")

    # Precision plot
    plt.subplot(6, 1, 5)
    plot_metric(epochs, train_precisions, val_precisions, "Precision")

    # Recall plot
    plt.subplot(6, 1, 6)
    plot_metric(epochs, train_recalls, val_recalls, "Recall")

    plt.tight_layout()
    plt.savefig(f'{config.base_path_model}{config.model_iteration}/cv_results.png')
    plt.close()

def plot_metric(epochs, train_metric, val_metric, metric_name):
    train_mean = np.mean(train_metric, axis=0)
    train_std = np.std(train_metric, axis=0)
    val_mean = np.mean(val_metric, axis=0)
    val_std = np.std(val_metric, axis=0)

    plt.plot(epochs, train_mean, 'b-', label=f'Training {metric_name}')
    plt.fill_between(epochs, train_mean - train_std, train_mean + train_std, alpha=0.1, color='b')
    plt.plot(epochs, val_mean, 'r-', label=f'Validation {metric_name}')
    plt.fill_between(epochs, val_mean - val_std, val_mean + val_std, alpha=0.1, color='r')
    
    # Optionally, plot individual fold lines
    # for i in range(train_metric.shape[0]):
    #     plt.plot(epochs, train_metric[i], 'b-', alpha=0.1)
    #     plt.plot(epochs, val_metric[i], 'r-', alpha=0.1)

    plt.title(f'Training and Validation {metric_name}')
    plt.xlabel('Epochs')
    plt.ylabel(metric_name)
    plt.legend()


# custom collate function for handling any remanining inconsistencies with DataLoader
def custom_collate(batch):
    batch = list(filter(lambda x: x is not None, batch))
    if len(batch) == 0:
        return None
    return torch.utils.data.dataloader.default_collate(batch)

def stratified_k_fold_cross_validation(config, full_dataset, transforms):
    """Performs stratified k-fold cross-validation."""    
    # Get stratification labels from cache if available
    labels, cache_used = get_cached_stratification_labels(config, full_dataset)
    
    if cache_used:
        print("Using cached stratification labels - this will be much faster!")

    # Create labels for stratification
    all_indices = np.arange(len(full_dataset))
    
    labels = []
    for i in tqdm(all_indices, desc="Generating Stratification Labels"):
        try:
            _, mask = full_dataset[i]
            if torch.sum(mask == 1) > 0:
                labels.append(1)  # Trash present
            else:
                labels.append(0)  # No trash
        except Exception as e:
            print(f"Error processing item {i}: {str(e)}")
            labels.append(0)  # Treat as no trash

    labels = np.array(labels)
    
    # Split into train+val and hold-out sets
    train_val_indices, holdout_indices, train_val_labels, holdout_labels = train_test_split(
        all_indices, labels, test_size=config.holdout_percentage, stratify=labels, random_state=config.random_seed
    )

    # Use StratifiedKFold
    skf = StratifiedKFold(n_splits=config.k_folds, shuffle=True, random_state=config.random_seed)
                          
    best_mcc_overall = -1.0
    best_model_state_overall = None
    best_fold = -1
    best_epoch = -1

    # Initialize lists to store metrics across all folds
    all_train_losses, all_train_f1_scores, all_train_mcc_scores, all_train_accuracies = [], [], [], []
    all_train_precisions, all_train_recalls = [], []
    all_val_losses, all_val_f1_scores, all_val_mcc_scores, all_val_accuracies = [], [], [], []
    all_val_precisions, all_val_recalls = [], []

    # Create a single progress bar for all folds
    for fold, (train_index, val_index) in enumerate(skf.split(train_val_indices, train_val_labels)):
        print(f"\n--- Fold {fold + 1}/{config.k_folds} ---")

        train_indices = train_val_indices[train_index]
        val_indices = train_val_indices[val_index]

        train_dataset = DroneTrashDataset(config, transforms, mode='train', indices=train_indices)
        val_dataset = DroneTrashDataset(config, transforms, mode='val', indices=val_indices)

        # Identify positive and negative samples
        train_pos = [i for i, idx in enumerate(train_indices) if labels[idx] == 1]
        train_neg = [i for i, idx in enumerate(train_indices) if labels[idx] == 0]
        val_pos = [i for i, idx in enumerate(val_indices) if labels[idx] == 1]
        val_neg = [i for i, idx in enumerate(val_indices) if labels[idx] == 0]

        train_curriculum = CurriculumDataset(train_dataset, train_pos, train_neg, config.num_epochs, mode='train', config=config)
        val_curriculum = CurriculumDataset(val_dataset, val_pos, val_neg, config.num_epochs, mode='val', config=config)

        train_loader = DataLoader(train_curriculum, batch_size=config.batch_size, shuffle=True, num_workers=0, pin_memory=True, collate_fn=custom_collate)
        val_loader = DataLoader(val_curriculum, batch_size=config.batch_size, shuffle=False, num_workers=0, pin_memory=True, collate_fn=custom_collate)

        # Reset model
        model.load_state_dict(torch.load(config.initial_model_weights))

        # Reset optimizer
        optimizer = optim.Adam(model.parameters(), lr=config.initial_lr, weight_decay=config.weight_decay)

        # Reset scheduler
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

        # Log data distribution
        print(f"Train set size: {len(train_indices)}, Validation set size: {len(val_indices)}")
        print(f"Positive samples in train: {len(train_pos)}, in validation: {len(val_pos)}")

        (fold_train_losses, fold_train_f1_scores, fold_train_mcc_scores, fold_train_accuracies,
         fold_train_precisions, fold_train_recalls,
         fold_val_losses, fold_val_f1_scores, fold_val_mcc_scores, fold_val_accuracies,
         fold_val_precisions, fold_val_recalls,
         fold_best_mcc, fold_best_model_state, fold_best_epoch) = train_model(model, config, train_loader, val_loader, criterion, optimizer, scheduler)

        all_train_losses.extend(fold_train_losses)
        all_train_f1_scores.extend(fold_train_f1_scores)
        all_train_mcc_scores.extend(fold_train_mcc_scores)
        all_train_accuracies.extend(fold_train_accuracies)
        all_train_precisions.extend(fold_train_precisions)
        all_train_recalls.extend(fold_train_recalls)
        all_val_losses.extend(fold_val_losses)
        all_val_f1_scores.extend(fold_val_f1_scores)
        all_val_mcc_scores.extend(fold_val_mcc_scores)
        all_val_accuracies.extend(fold_val_accuracies)
        all_val_precisions.extend(fold_val_precisions)
        all_val_recalls.extend(fold_val_recalls)

        if fold_best_mcc > best_mcc_overall:
            best_mcc_overall = fold_best_mcc
            best_model_state_overall = fold_best_model_state
            best_fold = fold + 1  # +1 because fold is 0-indexed
            best_epoch = fold_best_epoch
            tqdm.write(f"New best overall MCC: {best_mcc_overall:.6f} achieved in fold {best_fold} at epoch {best_epoch}. Saving model...")
            torch.save(best_model_state_overall, f'{config.base_path_model}{config.model_iteration}/unet_trash_detection_best.pth')

    # Plot the performance metrics across all epochs and folds
    plot_cv_results(all_train_losses, all_train_f1_scores, all_train_mcc_scores, all_train_accuracies,
                    all_train_precisions, all_train_recalls,
                    all_val_losses, all_val_f1_scores, all_val_mcc_scores, all_val_accuracies,
                    all_val_precisions, all_val_recalls,
                    config.k_folds, config.num_epochs, config)
    
    # Print final summary
    tqdm.write(f"\nTraining completed. Best model performance:")
    tqdm.write(f"Best MCC: {best_mcc_overall:.6f}")
    tqdm.write(f"Achieved in fold {best_fold} at epoch {best_epoch}")

    # After k-fold cross-validation, evaluate on hold-out set
    holdout_dataset = DroneTrashDataset(config, transforms, mode='test', indices=holdout_indices)
    holdout_loader = DataLoader(holdout_dataset, batch_size=config.batch_size, shuffle=False, num_workers=0, pin_memory=True, collate_fn=custom_collate)

    # Load the best model
    best_model = eval(config.model_def).to(device)
    best_model.load_state_dict(torch.load(f'{config.base_path_model}{config.model_iteration}/unet_trash_detection_best.pth'))

    # Evaluate on hold-out set
    holdout_loss, holdout_accuracy, holdout_precision, holdout_recall, holdout_f1, holdout_mcc = evaluate_model(
        best_model, holdout_loader, criterion, config, save_plots=True,
        plot_dir=f'{config.base_path_model}{config.model_iteration}',
        evaluation_type="holdout",
        is_best_epoch=True  # Always save plots for holdout
    )

    print(f"\nHold-out Set Performance:")
    print(f"Loss: {holdout_loss:.4f}")
    print(f"Accuracy: {holdout_accuracy:.4f}")
    print(f"Precision: {holdout_precision:.4f}")
    print(f"Recall: {holdout_recall:.4f}")
    print(f"F1 Score: {holdout_f1:.4f}")
    print(f"MCC: {holdout_mcc:.4f}")

    return best_model_state_overall, best_mcc_overall, best_fold, best_epoch, (holdout_loss, holdout_f1, holdout_mcc, holdout_accuracy)

    
def process_item(i, config, transform, mode):
    try:
        # Create a new DroneTrashDataset for each item
        dataset = DroneTrashDataset(config, transform, mode)
        item = dataset[i]
        if item is not None and torch.sum(item[1] == 1) > 0:
            return i
    except Exception as e:
        print(f"Error processing item {i}: {str(e)}")
    return None

def print_iteration_info(config):
    """Print information about the current model iteration."""
    print("=" * 60)
    print(f"Starting new model iteration: {config.model_iteration}")
    print(f"Model directory: {config.base_path_model + config.model_iteration}")
    print(f"Timestamp: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 60)

if __name__ == "__main__":
    try:
        # Get transforms before the dataset
        transforms = get_transforms(config)

        # Set this to False to use only positive samples
        config.use_negative_samples = True
    
        # Create the full dataset FIRST
        full_dataset = DroneTrashDataset(config, transforms, mode='base')
        print(f"Found {len(full_dataset.image_mask_pairs)} image-mask pairs:")
        for i, (img, mask) in enumerate(full_dataset.image_mask_pairs):
            print(f"Pair {i+1}: Image: {os.path.basename(img)}, Mask: {os.path.basename(mask)}")
        print(f"Total tiles: {full_dataset.total_tiles}, Valid tiles: {len(full_dataset.indices)}")

        # Check for existing cache files (we always create new directories)
        cache_available, cache_file_path = check_existing_model_iteration(config, full_dataset)
        
        # NOW determine the model iteration (only once!)
        config.model_iteration = get_next_model_iteration(config.base_path_model, config.base_iteration_name)
        
        # Update all the file paths based on the model iteration
        config.set_model_iteration(config.model_iteration)
        
        print(f"Creating new model iteration: {config.model_iteration}")
        
        # Print iteration information
        print_iteration_info(config)

        # Create directory for this model iteration
        Path(config.base_path_model + config.model_iteration).mkdir(parents=True, exist_ok=True)
        
        # Save configuration parameters to file
        save_config_to_file(config, config.base_path_model + config.model_iteration)
        
        # Save initial model weights
        torch.save(model.state_dict(), config.initial_model_weights)

        # Start k-fold cross-validation and get hold-out set performance
        best_model_state, best_mcc, best_fold, best_epoch, holdout_metrics = stratified_k_fold_cross_validation(config, full_dataset, transforms)

    except KeyboardInterrupt:
        print("Process interrupted by user. Exiting...")
        sys.exit(0)
    except Exception as e:
        print(f"Unexpected error in main execution: {e}")
        raise e

Base dataset - Total tiles: 356265, Valid tiles: 356265
Found 1 image-mask pairs:
Pair 1: Image: Makassar_Tallo_June2024_image.tif, Mask: Makassar_Tallo_June2024_mask_tires.tif
Total tiles: 356265, Valid tiles: 356265
Found cache file with matching tiling parameters from: makassar_tallo_june2024_tires_04
Will reuse cache file for faster evaluation...
Creating new model iteration: makassar_tallo_june2024_tires_08
Starting new model iteration: makassar_tallo_june2024_tires_08
Model directory: K:/dengue/unet_model_data/makassar_tallo_june2024_tires_08
Timestamp: 2025-08-23 19:02:35
Configuration parameters saved to: K:/dengue/unet_model_data/makassar_tallo_june2024_tires_08\config_parameters.txt
Using cached stratification labels from: K:/dengue/unet_model_data/stratification_cache\stratification_labels_3908f0192ee0bfa23776f8bfcb240a92.pkl
Cache info: 356265 labels, positive samples: 714, negative samples: 355551
Using cached stratification labels - this will be much faster!


Generating Stratification Labels: 100%|██████████| 356265/356265 [2:26:19<00:00, 40.58it/s]  



--- Fold 1/2 ---
Train dataset - Total tiles: 356265, Valid tiles: 160319
Val dataset - Total tiles: 356265, Valid tiles: 160319
Epoch 1 (Train): 322 positive, 32 negative samples (ratio: 0.10)
Epoch 1 (Val): 321 positive, 32 negative samples (ratio: 0.10)


C:\Users\andyc\AppData\Local\Temp\ipykernel_21084\336062107.py:1185: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(config.initial_model_weig

Train set size: 160319, Validation set size: 160319
Positive samples in train: 322, in validation: 321
Epoch 1 (Train): 322 positive, 32 negative samples (ratio: 0.10)
Epoch 1 (Val): 321 positive, 32 negative samples (ratio: 0.10)


...

New best model at epoch 2 with MCC: 8508.121967 and loss: 1.685535
Epoch 2/3 - Train Loss: 1.8078, Val Loss: 1.6855, Train F1: 0.1178, Val F1: 0.2016, Train MCC: 0.1012, Val MCC: 8508.1220, Train Precision: 0.0843, Val Precision: 0.1158, Train Recall: 0.2326, Val Recall: 0.7808, LR: 0.000100
Epoch 3 (Train): 321 positive, 32 negative samples (ratio: 0.10)
Epoch 3 (Val): 322 positive, 32 negative samples (ratio: 0.10)


C:\Users\andyc\AppData\Local\Temp\ipykernel_21084\336062107.py:924: RuntimeWarning: overflow encountered in scalar multiply
  denominator = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
C:\Users\andyc\AppData\Local\Temp\ipykernel_21084\336062107.py:924: RuntimeWarning: invalid value encountered in sqrt
  denominator = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))


Epoch 3/3 - Train Loss: 1.6674, Val Loss: 1.4452, Train F1: 0.2664, Val F1: 0.3483, Train MCC: 0.2885, Val MCC: 0.0000, Train Precision: 0.1760, Val Precision: 0.2176, Train Recall: 0.5880, Val Recall: 0.8721, LR: 0.000100

Training completed. Best model performance:
Best MCC: 21000.824439
Achieved in fold 1 at epoch 2
Test dataset - Total tiles: 356265, Valid tiles: 35627


C:\Users\andyc\AppData\Local\Temp\ipykernel_21084\336062107.py:1242: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  best_model.load_state_dict(torch.load(f'{config.base_path_


Error processing batch 1036: 

Error processing batch 1037: 

Error processing batch 1038: 

Error processing batch 1039: 

Error processing batch 1040: 

Error processing batch 1041: 

Error processing batch 1042: 

Error processing batch 1043: 

Error processing batch 1044: 

Error processing batch 1045: 

Error processing batch 1046: 

Error processing batch 1047: 

Error processing batch 1048: 

Error processing batch 1049: 

Error processing batch 1050: 

Error processing batch 1051: 

Error processing batch 1052: 

Error processing batch 1053: 

Error processing batch 1054: 

Error processing batch 1055: 

Error processing batch 1056: 

Error processing batch 1057: 

Error processing batch 1058: 

Error processing batch 1059: 

Error processing batch 1060: 

Error processing batch 1061: 

Error processing batch 1062: 

Error processing batch 1063: 

Error processing batch 1064: 

Error processing batch 1065: 

Error processing batch 1066: 

Error processing batch 1067: 

Error p